# Text Chunking, Embedding, and Vector Store Indexing

In [6]:
import sys
import os
sys.path.append(os.path.abspath('../'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src.embedder import (
    stratified_sample, chunk_dataframe, load_embedding_model,
    embed_chunks, build_chroma_store, CHUNK_SIZE, CHUNK_OVERLAP
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## Load Cleaned Dataset

In [3]:
df = pd.read_csv('../data/processed/filtered_complaints.csv')
print(f'Loaded {len(df):,} records')
print(df['Product'].value_counts())

Loaded 431,353 records
Product
Credit Card        176608
Savings Account    140319
Money Transfer      97188
Personal Loan       17238
Name: count, dtype: int64


# Stratified Sample

In [9]:
SAMPLE_SIZE = 2000
df_sample = stratified_sample(df, n=SAMPLE_SIZE)
print(f'Sample size: {len(df_sample):,}')
print('\nCategory distribution in sample:')
print(df_sample['product_category'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['product_category'].value_counts().plot(kind='bar', ax=axes[0], title='Full Dataset', color='steelblue')
df_sample['product_category'].value_counts().plot(kind='bar', ax=axes[1], title='Stratified Sample', color='coral')
for ax in axes:
    ax.set_xlabel('Product')
    ax.set_ylabel('Count')
    plt.sca(ax)
    plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/sample_distribution.png', dpi=120)
plt.show()

KeyError: 'product_category'

## Chenk Narrative

In [ ]:
print(f'Chunk size: {CHUNK_SIZE} chars | Overlap: {CHUNK_OVERLAP} chars')

chunks = chunk_dataframe(df_sample, text_col='clean_narrative')
print(f'Total chunks: {len(chunks):,}')
print(f'Avg chunks per complaint: {len(chunks)/len(df_sample):.1f}')

# Chunk length distribution
chunk_lengths = [len(c['text']) for c in chunks]
pd.Series(chunk_lengths).describe()

In [ ]:
# Show a sample chunk with metadata
sample_chunk = chunks[42]
print('Text:')
print(sample_chunk['text'])
print('\nMetadata:')
print(sample_chunk['metadata'])

## Embed Chunks

In [ ]:
print('Loading embedding model: all-MiniLM-L6-v2 ...')
model = load_embedding_model()
print('Model loaded.')

print(f'Embedding {len(chunks):,} chunks ...')
embeddings = embed_chunks(chunks, model, batch_size=128)
print(f'Embedding matrix shape: {embeddings.shape}')

## Build and Persist ChromaDB Store

In [ ]:
collection = build_chroma_store(
    chunks=chunks,
    embeddings=embeddings,
    persist_dir='../vector_store/chroma',
)
print(f'Collection count: {collection.count():,} chunks stored')

## Smoke Test - Semantic Search

In [ ]:
from src.embedder import query_store

test_question = 'Why are customers complaining about credit card billing?'
hits = query_store(collection, test_question, model, k=3)

for i, hit in enumerate(hits, 1):
    print(f'--- Hit {i} (distance={hit["distance"]:.4f}) ---')
    print(f'Product: {hit["metadata"].get("product_category")}')
    print(hit['text'][:300])
    print()